# 04: Literature Context for the Derived RR Lyrae Period-Luminosity Relations

This notebook places the class-separated Gaia $G$-band period-luminosity relations from `02-02.ipynb` and the class-separated WISE $W2$ relations from `03.ipynb` into a literature context. The notebook remains **optical-first**: the main question is how the derived Gaia $G$ slopes compare to the classical optical expectation that RR Lyrae PL relations are weak in the visual, while the second half uses the `W2` result to show how the relation changes as the bandpass moves into the infrared ([Klein et al. 2014](https://academic.oup.com/mnrasl/article/440/1/L96/1396776); [Beaton et al. 2018](https://doi.org/10.1007/s11214-018-0542-1)).

Two caveats matter up front. First, Gaia $G$ is not Johnson $V$: it is a broader and redder optical band, so a one-to-one coefficient match is not expected. Second, the upstream notebooks fit direct absolute magnitudes inferred from parallaxes, whereas much of the Gaia-era literature prefers parallax-space or hierarchical calibration because it controls bias more carefully ([Luri et al. 2018](https://www.aanda.org/articles/aa/full_html/2018/08/aa32964-18/aa32964-18.html); [Muraveva et al. 2018](https://academic.oup.com/mnras/article/481/1/1195/5075596)).

In [1]:
import io
import os
from contextlib import redirect_stdout
from pathlib import Path
from IPython.display import Markdown, display

cache_root = Path('/tmp/ay128_lab01_cache')
cache_root.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(cache_root / 'mpl'))
os.environ.setdefault('XDG_CACHE_HOME', str(cache_root / 'xdg'))
os.environ.setdefault('PYTENSOR_FLAGS', f"base_compiledir={cache_root / 'pytensor'}")

from astropy import table
import numpy as np

with redirect_stdout(io.StringIO()):
    from ugdatalab import (
        load_infrared_pl_comparison_data,
        load_optical_pl_comparison_data,
    )

optical_path = Path('rrlyrae_optical_pl_comparison_data.npz')
infrared_path = Path('rrlyrae_infrared_pl_comparison_data.npz')

optical_data = load_optical_pl_comparison_data(optical_path)
infrared_data = load_infrared_pl_comparison_data(infrared_path)

literature = {
    'klein_optical': {
        'statement': 'In optical wavebands the RR Lyrae period-luminosity relation is negligible, but the slope steepens strongly in the infrared.',
        'source': 'Klein et al. 2014, Introduction',
        'url': 'https://academic.oup.com/mnrasl/article/440/1/L96/1396776',
    },
    'beaton_review': {
        'statement': 'RR Lyrae calibration depends on wavelength, reddening, metallicity, and how the absolute scale is established.',
        'source': 'Beaton et al. 2018',
        'url': 'https://doi.org/10.1007/s11214-018-0542-1',
    },
    'klein_w2': {
        'RRab': {'slope': -2.39, 'err': 0.20, 'pivot': 0.55},
        'RRc': {'slope': -1.70, 'err': 0.62, 'pivot': 0.32},
        'url': 'https://academic.oup.com/mnrasl/article/440/1/L96/1396776',
    },
    'dambis_w2': {
        'slope': -2.269,
        'err': 0.127,
        'metallicity_coeff': 0.108,
        'metallicity_err': 0.021,
        'url': 'https://doi.org/10.1093/mnras/stu226',
    },
}

{
    'optical_artifact': str(optical_path.resolve()),
    'infrared_artifact': str(infrared_path.resolve()),
    'RRab_G_points': int(len(optical_data['RRab'].x_obs)),
    'RRc_G_points': int(len(optical_data['RRc'].x_obs)),
    'RRab_W2_points': int(len(infrared_data['RRab'].x_obs)),
    'RRc_W2_points': int(len(infrared_data['RRc'].x_obs)),
}

{'optical_artifact': '/Users/junruiting/GitHub/ay-128/labs/01/rrlyrae_optical_pl_comparison_data.npz',
 'infrared_artifact': '/Users/junruiting/GitHub/ay-128/labs/01/rrlyrae_infrared_pl_comparison_data.npz',
 'RRab_G_points': 560,
 'RRc_G_points': 315,
 'RRab_W2_points': 527,
 'RRc_W2_points': 304}

## Derived Relation Summaries

The two upstream artifacts already contain the posterior summaries that matter for literature comparison: the fitted slope, its 68% interval, and the intrinsic scatter. The table below puts the optical and infrared relations on the same footing before any interpretation.

In [2]:
derived_rows = []
for band_label, comparison_map in [('Gaia $G$', optical_data), ('WISE $W2$', infrared_data)]:
    for rr_class in ('RRab', 'RRc'):
        fit = comparison_map[rr_class]
        derived_rows.append(
            {
                'band': band_label,
                'class': rr_class,
                'N_fit': int(len(fit.x_obs)),
                'slope_q50': round(float(fit.slope_q50), 3),
                'slope_68pct': f"[{fit.slope_q16:.3f}, {fit.slope_q84:.3f}]",
                'sigma_scatter_q50': round(float(fit.sigma_scatter_q50), 3),
                'sigma_scatter_68pct': f"[{fit.sigma_scatter_q16:.3f}, {fit.sigma_scatter_q84:.3f}]",
            }
        )

derived_table = table.Table(rows=derived_rows)
derived_table

band,class,N_fit,slope_q50,slope_68pct,sigma_scatter_q50,sigma_scatter_68pct
str9,str4,int64,float64,str16,float64,str14
Gaia $G$,RRab,560,-2.232,"[-2.359, -2.102]",0.176,"[0.169, 0.184]"
Gaia $G$,RRc,315,-2.209,"[-2.391, -2.026]",0.166,"[0.157, 0.176]"
WISE $W2$,RRab,527,-2.984,"[-3.094, -2.872]",0.147,"[0.141, 0.155]"
WISE $W2$,RRc,304,-3.246,"[-3.402, -3.091]",0.133,"[0.125, 0.142]"


## Literature Benchmarks

The optical benchmark is intentionally conservative. [Klein et al. (2014)](https://academic.oup.com/mnrasl/article/440/1/L96/1396776) emphasize that the RR Lyrae PL relation is **negligible in the optical** and much steeper in the infrared, so the classical Johnson $V$ comparison point is effectively a near-zero period term rather than a strong optical PL slope. [Beaton et al. (2018)](https://doi.org/10.1007/s11214-018-0542-1) then explain why the answer depends on wavelength, reddening, metallicity, and calibration strategy.

For the infrared section, the notebook uses the subclass-separated `W2` slopes reported by [Klein et al. (2014)](https://academic.oup.com/mnrasl/article/440/1/L96/1396776), and one additional WISE benchmark from [Dambis et al. (2014)](https://doi.org/10.1093/mnras/stu226), who fit a fundamentalized mixed-sample $W2$ period-luminosity-metallicity relation.

In [3]:
literature_rows = [
    {
        "reference": "Klein et al. 2014 (inferred visual benchmark)",
        "band_or_scope": "Johnson V / classical optical context",
        "benchmark": "adopted a_V ≈ 0.0 mag dex^-1",
        "use_here": "Concrete numerical proxy for the paper statement that the optical RR Lyrae PL slope is negligible.",
    },
    {
        'reference': 'Klein et al. 2014',
        'band_or_scope': 'Classical optical / visual context',
        'benchmark': 'Optical PL relation is negligible; infrared slopes steepen strongly.',
        'use_here': 'Treats Johnson V as a weak-period benchmark rather than a like-for-like Gaia G coefficient.',
    },
    {
        'reference': 'Beaton et al. 2018',
        'band_or_scope': 'Population II distance-indicator review',
        'benchmark': 'Calibration depends on bandpass, reddening, metallicity, and zero-point strategy.',
        'use_here': 'Explains why systematic offsets from our simple Gaia G fit are expected.',
    },
    {
        'reference': 'Klein et al. 2014',
        'band_or_scope': 'WISE W2, RRab',
        'benchmark': 'a_W2 = -2.39 +/- 0.20',
        'use_here': 'Subclass-matched W2 slope benchmark for RRab.',
    },
    {
        'reference': 'Klein et al. 2014',
        'band_or_scope': 'WISE W2, RRc',
        'benchmark': 'a_W2 = -1.70 +/- 0.62',
        'use_here': 'Subclass-matched W2 slope benchmark for RRc.',
    },
    {
        'reference': 'Dambis et al. 2014',
        'band_or_scope': 'WISE W2, mixed PLZ relation',
        'benchmark': '<M_W2> = gamma_W2 - (2.269 +/- 0.127) log P_F + (0.108 +/- 0.021) [Fe/H]',
        'use_here': 'Additional infrared reference showing that metallicity remains relevant even in W2.',
    },
]

literature_table = table.Table(rows=literature_rows)
literature_table

reference,band_or_scope,benchmark,use_here
str18,str39,str81,str91
Klein et al. 2014,Classical optical / visual context,Optical PL relation is negligible; infrared slopes steepen strongly.,Treats Johnson V as a weak-period benchmark rather than a like-for-like Gaia G coefficient.
Beaton et al. 2018,Population II distance-indicator review,"Calibration depends on bandpass, reddening, metallicity, and zero-point strategy.",Explains why systematic offsets from our simple Gaia G fit are expected.
Klein et al. 2014,"WISE W2, RRab",a_W2 = -2.39 +/- 0.20,Subclass-matched W2 slope benchmark for RRab.
Klein et al. 2014,"WISE W2, RRc",a_W2 = -1.70 +/- 0.62,Subclass-matched W2 slope benchmark for RRc.
Dambis et al. 2014,"WISE W2, mixed PLZ relation",<M_W2> = gamma_W2 - (2.269 +/- 0.127) log P_F + (0.108 +/- 0.021) [Fe/H],Additional infrared reference showing that metallicity remains relevant even in W2.


## Optical Context: Gaia $G$ in the Classical Optical Literature

This section keeps the notebook optical-first, but the comparison is made numerically rather than graphically. The relevant literature point is not a strict same-band overlay: the classical optical benchmark is that the visual-band RR Lyrae period-luminosity relation is weak, while Gaia $G$ is a broader and somewhat redder optical band ([Klein et al. 2014](https://academic.oup.com/mnrasl/article/440/1/L96/1396776); [Beaton et al. 2018](https://doi.org/10.1007/s11214-018-0542-1)).

In [4]:
rrab_g = optical_data['RRab']
rrc_g = optical_data['RRc']

optical_discussion = f"""
## Analysis: Gaia $G$ Versus the Classical Optical Literature

Our fitted Gaia $G$ relations are clearly non-zero: for RRab we find $a_G = {rrab_g.slope_q50:.3f}$ with a 68% interval of [{rrab_g.slope_q16:.3f}, {rrab_g.slope_q84:.3f}], and for RRc we find $a_G = {rrc_g.slope_q50:.3f}$ with a 68% interval of [{rrc_g.slope_q16:.3f}, {rrc_g.slope_q84:.3f}]. Those slopes are much stronger than the classical visual-band benchmark adopted here, $a_V \approx 0.0$ mag dex$^{-1}$, which I use as a numerical proxy for the [Klein et al. (2014)](https://academic.oup.com/mnrasl/article/440/1/L96/1396776) statement that the optical RR Lyrae PL slope is negligible and only becomes strongly constrained toward the infrared.

That difference is not automatically a contradiction. The first reason is bandpass. Gaia $G$ is not Johnson $V$; it is broader and reaches farther to the red, so it can naturally sit between the nearly-flat visual benchmark and the much steeper infrared behavior. The second reason is model specification. [Beaton et al. (2018)](https://doi.org/10.1007/s11214-018-0542-1) review the fact that RR Lyrae calibration depends on wavelength, reddening, metallicity, and the way the absolute scale is established. Our upstream fit is intentionally simple: it is a period-luminosity model with intrinsic scatter, but no explicit metallicity term and no hierarchical treatment of the absolute scale. Any true PLZ dependence can therefore leak into both the fitted slope and the residual scatter.

The third reason is methodology. The upstream notebook works in direct absolute-magnitude space after converting Gaia parallaxes into $M_G$, whereas Gaia-era calibration papers generally recommend parallax-space or hierarchical Bayesian treatments because they better control inverse-parallax bias and zero-point systematics ([Luri et al. 2018](https://www.aanda.org/articles/aa/full_html/2018/08/aa32964-18/aa32964-18.html); [Muraveva et al. 2018](https://academic.oup.com/mnras/article/481/1/1195/5075596)). A fourth reason is sample construction: our Gaia sample is local, quality-cut, and explicitly split into RRab and RRc, while many published optical comparisons use different extinction corrections, metallicity baselines, or mixed/fundamentalized samples.

The defensible scientific conclusion is therefore modest. Our Gaia $G$ slopes should **not** be read as direct replacements for a Johnson $V$ PL coefficient. They should instead be read as evidence that a broad Gaia optical band already begins to pick up some of the stronger period dependence that becomes cleaner in the infrared. That interpretation is consistent with the multiwavelength picture in [Klein et al. (2014)](https://academic.oup.com/mnrasl/article/440/1/L96/1396776) and the calibration caveats emphasized by [Beaton et al. (2018)](https://doi.org/10.1007/s11214-018-0542-1).
"""

display(Markdown(optical_discussion))


## Analysis: Gaia $G$ Versus the Classical Optical Literature

Our fitted Gaia $G$ relations are clearly non-zero: for RRab we find $a_G = -2.232$ with a 68% interval of [-2.359, -2.102], and for RRc we find $a_G = -2.209$ with a 68% interval of [-2.391, -2.026]. Those slopes are much stronger than the classical visual-band expectation summarized by [Klein et al. (2014)](https://academic.oup.com/mnrasl/article/440/1/L96/1396776), who note that in optical wavebands the RR Lyrae PL slope is negligible and only becomes strongly constrained toward the infrared.

That difference is not automatically a contradiction. The first reason is bandpass. Gaia $G$ is not Johnson $V$; it is broader and reaches farther to the red, so it can naturally sit between the nearly-flat visual benchmark and the much steeper infrared behavior. The second reason is model specification. [Beaton et al. (2018)](https://doi.org/10.1007/s11214-018-0542-1) review the fact that RR Lyrae calibration depends on wavelength, reddening, metallicity, and the way the absolute scale is established. Our upstream fit is intentionally simple: it is a period-luminosity model with intrinsic scatter, but no explicit metallicity term and no hierarchical treatment of the absolute scale. Any true PLZ dependence can therefore leak into both the fitted slope and the residual scatter.

The third reason is methodology. The upstream notebook works in direct absolute-magnitude space after converting Gaia parallaxes into $M_G$, whereas Gaia-era calibration papers generally recommend parallax-space or hierarchical Bayesian treatments because they better control inverse-parallax bias and zero-point systematics ([Luri et al. 2018](https://www.aanda.org/articles/aa/full_html/2018/08/aa32964-18/aa32964-18.html); [Muraveva et al. 2018](https://academic.oup.com/mnras/article/481/1/1195/5075596)). A fourth reason is sample construction: our Gaia sample is local, quality-cut, and explicitly split into RRab and RRc, while many published optical comparisons use different extinction corrections, metallicity baselines, or mixed/fundamentalized samples.

The defensible scientific conclusion is therefore modest. Our Gaia $G$ slopes should **not** be read as direct replacements for a Johnson $V$ PL coefficient. They should instead be read as evidence that a broad Gaia optical band already begins to pick up some of the stronger period dependence that becomes cleaner in the infrared. That interpretation is consistent with the multiwavelength picture in [Klein et al. (2014)](https://academic.oup.com/mnrasl/article/440/1/L96/1396776) and the calibration caveats emphasized by [Beaton et al. (2018)](https://doi.org/10.1007/s11214-018-0542-1).


## Infrared Comparison: How the Relation Changes in $W2$

The next table compares the notebook's own `W2` result from `03.ipynb` against its Gaia $G$ result, then places those infrared slopes against the `W2` literature anchors. That numerical comparison is enough to show the physical trend emphasized in the literature: moving to the infrared should generally make the relation steeper and tighter, although the exact coefficients still depend on sample definition and calibration strategy ([Klein et al. 2014](https://academic.oup.com/mnrasl/article/440/1/L96/1396776); [Beaton et al. 2018](https://doi.org/10.1007/s11214-018-0542-1)).

In [5]:
comparison_rows = []
for rr_class in ('RRab', 'RRc'):
    g_fit = optical_data[rr_class]
    w2_fit = infrared_data[rr_class]
    klein_fit = literature['klein_w2'][rr_class]
    comparison_rows.append(
        {
            'class': rr_class,
            'G_slope': f"{g_fit.slope_q50:.3f} [{g_fit.slope_q16:.3f}, {g_fit.slope_q84:.3f}]",
            'W2_slope': f"{w2_fit.slope_q50:.3f} [{w2_fit.slope_q16:.3f}, {w2_fit.slope_q84:.3f}]",
            'W2_minus_G': round(float(w2_fit.slope_q50 - g_fit.slope_q50), 3),
            'Klein_W2_slope': f"{klein_fit['slope']:.2f} +/- {klein_fit['err']:.2f}",
            'W2_minus_Klein': round(float(w2_fit.slope_q50 - klein_fit['slope']), 3),
            'G_scatter': round(float(g_fit.sigma_scatter_q50), 3),
            'W2_scatter': round(float(w2_fit.sigma_scatter_q50), 3),
        }
    )

band_comparison_table = table.Table(rows=comparison_rows)
band_comparison_table

class,G_slope,W2_slope,W2_minus_G,Klein_W2_slope,W2_minus_Klein,G_scatter,W2_scatter
str4,str23,str23,float64,str14,float64,float64,float64
RRab,"-2.232 [-2.359, -2.102]","-2.984 [-3.094, -2.872]",-0.753,-2.39 +/- 0.20,-0.594,0.176,0.147
RRc,"-2.209 [-2.391, -2.026]","-3.246 [-3.402, -3.091]",-1.037,-1.70 +/- 0.62,-1.546,0.166,0.133


In [6]:
rrab_w2 = infrared_data['RRab']
rrc_w2 = infrared_data['RRc']

infrared_discussion = f"""
## Analysis: $W2$ Versus $G$, and $W2$ Versus the Infrared Literature

The internal comparison is clear. For RRab, the slope changes from $a_G = {rrab_g.slope_q50:.3f}$ to $a_{{W2}} = {rrab_w2.slope_q50:.3f}$, while the intrinsic scatter drops from {rrab_g.sigma_scatter_q50:.3f} mag to {rrab_w2.sigma_scatter_q50:.3f} mag. For RRc, the slope changes from $a_G = {rrc_g.slope_q50:.3f}$ to $a_{{W2}} = {rrc_w2.slope_q50:.3f}$, and the intrinsic scatter drops from {rrc_g.sigma_scatter_q50:.3f} mag to {rrc_w2.sigma_scatter_q50:.3f} mag. That is exactly the direction expected from the physical argument in [Klein et al. (2014)](https://academic.oup.com/mnrasl/article/440/1/L96/1396776): once the bandpass moves into the infrared, reduced extinction sensitivity and a weaker temperature term make the relation steeper and tighter.

Relative to the published `W2` literature, however, our infrared slopes are still systematically steeper. For RRab, [Klein et al. (2014)](https://academic.oup.com/mnrasl/article/440/1/L96/1396776) report $a_{{W2}} = -2.39 ± 0.20$, compared to our {rrab_w2.slope_q50:.3f}. For RRc, they report $a_{{W2}} = -1.70 ± 0.62$, compared to our {rrc_w2.slope_q50:.3f}. [Dambis et al. (2014)](https://doi.org/10.1093/mnras/stu226) also find a somewhat shallower mixed-sample infrared slope, together with a non-negligible metallicity term in $W2$. That combination is informative: it suggests that the broad optical-to-infrared trend is reproduced, but the exact coefficient still depends on how the calibration is built.

Several effects can drive the remaining offset. First, the samples are genuinely different. Our `W2` fit is built from the Gaia-cleaned and Gaia+WISE-matched sample used in this lab, whereas the Klein et al. calibration used 104 RRab stars and only 19 RRc stars with prior distances tied to an established $M_V$-[Fe/H] scale plus a handful of HST parallaxes. Second, the literature often either fundamentalizes RRc periods or mixes subclasses in a PLZ framework, while our notebook keeps RRab and RRc fully separated. Third, even in the infrared, metallicity has not disappeared; [Dambis et al. (2014)](https://doi.org/10.1093/mnras/stu226) explicitly fit a metallicity term, while our notebook does not. Fourth, the absolute scale is still being set differently across analyses, and [Beaton et al. (2018)](https://doi.org/10.1007/s11214-018-0542-1) stress that these zero-point and calibration choices remain a major systematic.

So the strongest conclusion is comparative rather than absolute. The notebook reproduces the key astrophysical trend: `W2` is steeper and tighter than Gaia $G$, consistent with the infrared literature. But the exact `W2` slopes are not identical to published calibrations, especially for RRc, and the likely explanation is a combination of bandpass, metallicity, subclass handling, sample construction, and calibration methodology rather than a single isolated failure of the fit.
"""

display(Markdown(infrared_discussion))


## Analysis: $W2$ Versus $G$, and $W2$ Versus the Infrared Literature

The internal comparison is clear. For RRab, the slope changes from $a_G = -2.232$ to $a_{W2} = -2.984$, while the intrinsic scatter drops from 0.176 mag to 0.147 mag. For RRc, the slope changes from $a_G = -2.209$ to $a_{W2} = -3.246$, and the intrinsic scatter drops from 0.166 mag to 0.133 mag. That is exactly the direction expected from the physical argument in [Klein et al. (2014)](https://academic.oup.com/mnrasl/article/440/1/L96/1396776): once the bandpass moves into the infrared, reduced extinction sensitivity and a weaker temperature term make the relation steeper and tighter.

Relative to the published `W2` literature, however, our infrared slopes are still systematically steeper. For RRab, [Klein et al. (2014)](https://academic.oup.com/mnrasl/article/440/1/L96/1396776) report $a_{W2} = -2.39 ± 0.20$, compared to our -2.984. For RRc, they report $a_{W2} = -1.70 ± 0.62$, compared to our -3.246. [Dambis et al. (2014)](https://doi.org/10.1093/mnras/stu226) also find a somewhat shallower mixed-sample infrared slope, together with a non-negligible metallicity term in $W2$. That combination is informative: it suggests that the broad optical-to-infrared trend is reproduced, but the exact coefficient still depends on how the calibration is built.

Several effects can drive the remaining offset. First, the samples are genuinely different. Our `W2` fit is built from the Gaia-cleaned and Gaia+WISE-matched sample used in this lab, whereas the Klein et al. calibration used 104 RRab stars and only 19 RRc stars with prior distances tied to an established $M_V$-[Fe/H] scale plus a handful of HST parallaxes. Second, the literature often either fundamentalizes RRc periods or mixes subclasses in a PLZ framework, while our notebook keeps RRab and RRc fully separated. Third, even in the infrared, metallicity has not disappeared; [Dambis et al. (2014)](https://doi.org/10.1093/mnras/stu226) explicitly fit a metallicity term, while our notebook does not. Fourth, the absolute scale is still being set differently across analyses, and [Beaton et al. (2018)](https://doi.org/10.1007/s11214-018-0542-1) stress that these zero-point and calibration choices remain a major systematic.

So the strongest conclusion is comparative rather than absolute. The notebook reproduces the key astrophysical trend: `W2` is steeper and tighter than Gaia $G$, consistent with the infrared literature. But the exact `W2` slopes are not identical to published calibrations, especially for RRc, and the likely explanation is a combination of bandpass, metallicity, subclass handling, sample construction, and calibration methodology rather than a single isolated failure of the fit.


### References

- [Beaton, R. L., Bono, G., Braga, V. F., et al. (2018), *Old-Aged Primary Distance Indicators*, Space Science Reviews, 214, 113.](https://doi.org/10.1007/s11214-018-0542-1)
- [Klein, C. R., Richards, J. W., Butler, N. R., & Bloom, J. S. (2014), *Mid-infrared period-luminosity relations of RR Lyrae stars derived from the AllWISE Data Release*, MNRAS Letters, 440, L96-L100.](https://academic.oup.com/mnrasl/article/440/1/L96/1396776)
- [Dambis, A. K., Rastorguev, A. S., & Zabolotskikh, M. V. (2014), *Mid-infrared period-luminosity relations for globular cluster RR Lyrae*, MNRAS, 439, 3765-3774.](https://doi.org/10.1093/mnras/stu226)
- [Luri, X., Brown, A. G. A., Sarro, L. M., et al. (2018), *Gaia Data Release 2: Using Gaia parallaxes*, A&A, 616, A9.](https://www.aanda.org/articles/aa/full_html/2018/08/aa32964-18/aa32964-18.html)
- [Muraveva, T., Delgado, H. E., Clementini, G., et al. (2018), *Gaia DR2 parallaxes and Bayesian RR Lyrae calibration*, MNRAS, 481, 1195-1211.](https://academic.oup.com/mnras/article/481/1/1195/5075596)